In [19]:
import onnx
from optimum.onnxruntime import ORTQuantizer, ORTModelForSequenceClassification
from functools import partial
from transformers import AutoTokenizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig, AutoCalibrationConfig, QuantizationConfig
from onnxruntime.quantization import QuantFormat, QuantizationMode, QuantType
from optimum.onnxruntime import ORTQuantizer
from optimum.onnxruntime.configuration import AutoCalibrationConfig, QuantizationConfig
from optimum.onnxruntime.modeling_ort import ORTModelForImageClassification
from optimum.onnxruntime.preprocessors import QuantizationPreprocessor
from optimum.onnxruntime.preprocessors.passes import (
    ExcludeGeLUNodes,
    ExcludeLayerNormNodes,
    ExcludeNodeAfter,
    ExcludeNodeFollowedBy,
)
from optimum.onnxruntime.utils import evaluation_loop
from datasets import load_dataset
from torchvision.transforms import CenterCrop, Compose, Normalize, Resize, ToTensor

import  torch
import numpy as np
from ultralytics.data.augment import LetterBox


class PreProcessor:
    def __init__(self, new_shape=(640,640), stride=32):
        self.letterbox = LetterBox(
            new_shape = new_shape,
            auto      = False, # always 640,640
            stride    = stride,
        )        
    def __call__(self, im):
        im = self.pre_transform(im)
        im = np.stack(im)
        im = im[..., ::-1].transpose((0, 3, 1, 2))
        im = np.ascontiguousarray(im)  # contiguous
        im = torch.from_numpy(im)
        im = im.float()
        im /= 255
        return im
    def pre_transform(self,im):
        result = [self.letterbox(image=x) for x in im]
        return result

def transform(data_batch, preprocessor):
    origin_shape=[]
    image_ = []
    data_batch['origin_image'] = data_batch['image']
    for im in data_batch['image']:
        im = np.array(im.convert('RGB'))[..., ::-1]
        image_.append(im)
        origin_shape.append(im.shape)
    image_ = preprocessor(image_)
    data_batch["image"] = image_
    data_batch['origin_shape'] = origin_shape
    return data_batch



In [2]:
quantizer = ORTQuantizer.from_pretrained('.')

Could not load the config for yolov8s.onnx automatically, this might make the quantized model harder to use because it will not be able to be loaded by an ORTModel without having to specify the configuration explicitly.


In [3]:
qconfig = QuantizationConfig(
is_static             = True,
format                = QuantFormat.QDQ,
mode                  = QuantizationMode.QLinearOps,
activations_dtype     = QuantType.QInt8,
weights_dtype         = QuantType.QInt8,
per_channel           = True,
reduce_range          = False,
operators_to_quantize = ["MatMul", "Conv", "Gemm"],
)

In [6]:
quantization_preprocessor = QuantizationPreprocessor()

In [21]:
dataset = load_dataset(path='rafaelpadilla/coco2017', cache_dir='/Data/Dataset/COCO', split='val').shuffle(seed=42)

Resolving data files:   0%|          | 0/39 [00:00<?, ?it/s]

In [ ]:
calibration_config = AutoCalibrationConfig.

Dataset({
    features: ['image', 'image_id', 'objects'],
    num_rows: 5000
})